# 05 - QpiAI Quantum / QCloud local simulator experiment

**Purpose:** Reproduce the reported -13.07 QAOA energy and apply the same QUBO formulation to the Phase 2 Wave-0 selection.

**Inputs:** qpiai-opt[quantum], qpiai-quantum; device QpiAI-QSV-Local (in-process statevector)

**Outputs:** reproduction result, Phase 2 QAOA sample statistics, counts chart

**Status labels:** SIMULATOR only. No QPU, no platform mode. No quantum advantage is claimed.

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
from cryptoshift.optimization.member3_qpiai import reproduce_reported_experiment
r = reproduce_reported_experiment(); {k: r[k] for k in ('local_qsv_simulator_status','best_simulator_wave0_selection','best_simulator_wave0_energy','reproduced','caveats')}

In [ ]:
from cryptoshift.planning.planner import build_problem
from cryptoshift.planning.dependency_graph import constraint_digraph
from cryptoshift.optimization.member3_qpiai import run_wave0_qaoa
p = build_problem(); c = p['config']
d = constraint_digraph(p['graph'], c['enforced_edge_provenance'])
cands = [{'id': s, 'reward': v['risk']['score'], 'minutes': v['duration_minutes']} for s, v in sorted(p['services'].items()) if v['verification_ready_wave0']]
ids = {x['id'] for x in cands}
q = run_wave0_qaoa(cands, sorted((u, v) for u, v in d.edges if u in ids and v in ids), c['constraints']['max_services_per_wave'], c['constraints']['max_minutes_per_wave'])
{k: q[k] for k in ('label','exact_optimum','most_frequent_sample','best_feasible_sample','optimum_probability','uniform_probability','feasible_shot_fraction','circuit')}

**Plain language.** The simulator draws 1024 samples from the QAOA circuit. With fixed angles and one layer, the best answer (the exact optimum) appears only a little more often than random guessing would give (compare `optimum_probability` with `uniform_probability`). The old experiment reported the best feasible sample, which almost any sampler finds on 4 variables. Bit order was measured: the simulator's strings are little-endian.